In [8]:
%pip install pandas requests

from pathlib import Path
from typing import Dict
import pandas as pd
import requests

SPLIT_URLS: Dict[str, Dict[str, str]] = {
    "train": {
        "sentences": "https://drive.google.com/uc?id=1nzak5OkrheRV1ltOGCXkT671bmjODLhP&export=download",
        "sentiments": "https://drive.google.com/uc?id=1ye-gOZIBqXdKOoi_YxvpT6FeRNmViPPv&export=download",
        "topics": "https://drive.google.com/uc?id=14MuDtwMnNOcr4z_8KdpxprjbwaQ7lJ_C&export=download",
    },
    "validation": {
        "sentences": "https://drive.google.com/uc?id=1sMJSR3oRfPc3fe1gK-V3W5F24tov_517&export=download",
        "sentiments": "https://drive.google.com/uc?id=1GiY1AOp41dLXIIkgES4422AuDwmbUseL&export=download",
        "topics": "https://drive.google.com/uc?id=1DwLgDEaFWQe8mOd7EpF-xqMEbDLfdT-W&export=download",
    },
    "test": {
        "sentences": "https://drive.google.com/uc?id=1aNMOeZZbNwSRkjyCWAGtNCMa3YrshR-n&export=download",
        "sentiments": "https://drive.google.com/uc?id=1vkQS5gI0is4ACU58-AbWusnemw7KZNfO&export=download",
        "topics": "https://drive.google.com/uc?id=1_ArMpDguVsbUGl-xSMkTF_p5KpZrmpSB&export=download",
    },
}

SENTIMENT_LABELS = {0: "negative", 1: "neutral", 2: "positive"}
TOPIC_LABELS = {0: "lecturer", 1: "training_program", 2: "facility", 3: "others"}

NOTEBOOK_DIR = Path.cwd()
CACHE_DIR = NOTEBOOK_DIR / "download_cache"
OUTPUT_DIR = (NOTEBOOK_DIR / "../processed").resolve()
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def download_text(url: str, destination: Path) -> Path:
    if destination.exists():
        return destination
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    destination.write_bytes(response.content)
    return destination


def build_split(split_name: str, urls: Dict[str, str]) -> Path:
    local_paths = {}
    for key, url in urls.items():
        file_path = CACHE_DIR / f"{split_name}_{key}.txt"
        local_paths[key] = download_text(url, file_path)

    with (
        local_paths["sentences"].open(encoding="utf-8") as sentences_file,
        local_paths["sentiments"].open(encoding="utf-8") as sentiments_file,
        local_paths["topics"].open(encoding="utf-8") as topics_file,
    ):
        rows = []
        for sentence, sentiment, topic in zip(sentences_file, sentiments_file, topics_file):
            sentiment_id = int(sentiment.strip())
            topic_id = int(topic.strip())
            rows.append(
                {
                    "sentence": sentence.strip(),
                    "sentiment_id": sentiment_id,
                    "sentiment": SENTIMENT_LABELS.get(sentiment_id, "unknown"),
                    "topic_id": topic_id,
                    "topic": TOPIC_LABELS.get(topic_id, "unknown"),
                }
            )

    dataframe = pd.DataFrame(rows)
    output_path = OUTPUT_DIR / f"{split_name}.csv"
    dataframe.to_csv(output_path, index=False)
    print(f"Saved {split_name} -> {output_path}")
    return output_path


saved_files = []
for split, urls in SPLIT_URLS.items():
    saved_files.append(build_split(split, urls))

print("Downloads complete:")
for path in saved_files:
    print(f" - {path}")

Note: you may need to restart the kernel to use updated packages.
Saved train -> F:\University of information technology's Courses\Xử lý ngôn ngữ tự nhiên\CS221_PrJ\datasets\processed\train.csv
Saved train -> F:\University of information technology's Courses\Xử lý ngôn ngữ tự nhiên\CS221_PrJ\datasets\processed\train.csv
Saved validation -> F:\University of information technology's Courses\Xử lý ngôn ngữ tự nhiên\CS221_PrJ\datasets\processed\validation.csv
Saved validation -> F:\University of information technology's Courses\Xử lý ngôn ngữ tự nhiên\CS221_PrJ\datasets\processed\validation.csv
Saved test -> F:\University of information technology's Courses\Xử lý ngôn ngữ tự nhiên\CS221_PrJ\datasets\processed\test.csv
Downloads complete:
 - F:\University of information technology's Courses\Xử lý ngôn ngữ tự nhiên\CS221_PrJ\datasets\processed\train.csv
 - F:\University of information technology's Courses\Xử lý ngôn ngữ tự nhiên\CS221_PrJ\datasets\processed\validation.csv
 - F:\University of